# IMPORT

In [12]:
import pandas as pd
from pathlib import Path
import joblib
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# DEFINE FOLDERS

In [13]:
STATION_CLUSTER_FOLDER = Path("station_clusters")
DATASET_FOLDER = Path("correlated_data_csv")
MODEL_OUTPUT_FOLDER = Path("cluster_models")
PLOT_FOLDER = Path("model_plots")

folders = [STATION_CLUSTER_FOLDER, DATASET_FOLDER, MODEL_OUTPUT_FOLDER, PLOT_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# CHOICE TO SPLIT THE DATA

In [14]:
WILL_SPLIT_DATA = True

# LOAD DATA AND CLUSTERS

In [15]:
df = pd.read_csv(f"{DATASET_FOLDER}/final_clean_data.csv") if WILL_SPLIT_DATA else pd.read_csv(f"{DATASET_FOLDER}/final_clean_split_data.csv")
clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/station_clusters_2_42.csv")

df = df.merge(clusters, on="station_code", how="inner")

print("Rows:", df.shape)
print("Clusters:", df["cluster"].value_counts())
print(df.columns)

Rows: (6803688, 18)
Clusters: cluster
0    3458726
1    3344962
Name: count, dtype: int64
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'cluster'],
      dtype='object')


# FEATURE SELECTION

In [16]:
FEATURES = [
    # Time
    "sin_hour",
    "cos_hour",
    "is_weekend",

    # Weather
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",

    # Station context
    "capacity"
]

TARGET = "total_bikes_now"


# TIME BASED 70-15-15 SPLIT

In [17]:
def time_split(df, train_frac=0.7, val_frac=0.15):
    """
    Gives indices of the data split
    """
    df = df.sort_values("timestamp")

    n = df.shape[0]
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    return train_end, val_end


# FREEZE THE DATASET AFTER SPLITTING

In [ ]:
df_c = None
if WILL_SPLIT_DATA:
    df["split"] = None
    df_c = None

    for cluster_id in df["cluster"].unique():
        mask = df["cluster"] == cluster_id
        df_c = df[mask].sort_values("timestamp")

        # Create lag features per station
        df_c["total_bikes_lag1"] = df_c.groupby("station_code")["total_bikes_now"].shift(1)
        df_c["total_bikes_lag2"] = df_c.groupby("station_code")["total_bikes_now"].shift(2)

        # Drop rows where lag is NaN
        df_c = df_c.dropna(subset=["total_bikes_lag1", "total_bikes_lag2"])

        # Compute indices for splitting
        train_end, val_end = time_split(df_c, 0.7, 0.15)

        # Assign split labels back to the original dataframe
        df_c.loc[df_c.index[:train_end], "split"] = "train"
        df_c.loc[df_c.index[train_end:val_end], "split"] = "val"
        df_c.loc[df_c.index[val_end:], "split"] = "test"
else:
    print("Dataset already split, skipping...")

# QUICK CHECK BEFORE SAVING THE FROZEN SPLIT DATASET

In [21]:
display(df.head(5))

,timestamp,station_code,latitude,longitude,capacity,total_bikes_now,total_bikes_taken,temperature,wind_speed,wind_direction,weather_code,is_day,hour,day_of_week,is_weekend,sin_hour,cos_hour,cluster,split
0,2025-11-04 04:00:00,1001,48.857059,2.341798,17.0,2.0,15.0,10.100000,9.500000,217.000000,3.000000,0.000000,4,1,0,0.866025,0.500000,1,None
1,2025-11-04 04:15:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.094372,9.283279,192.757663,14.660302,0.365327,4,1,0,0.866025,0.500000,1,None
2,2025-11-04 04:30:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.000000,8.000000,207.000000,3.000000,0.000000,4,1,0,0.866025,0.500000,1,train
3,2025-11-04 04:45:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.100000,8.200000,203.000000,3.000000,0.000000,4,1,0,0.866025,0.500000,1,train
4,2025-11-04 05:00:00,1001,48.857059,2.341798,17.0,2.0,15.0,9.094372,9.283279,192.757663,14.660302,0.365327,5,1,0,0.965926,0.258819,1,train


# SAVE THE FROZEN SPLIT DATASET

In [ ]:
if WILL_SPLIT_DATA:
    df_c.to_csv(f"{DATASET_FOLDER}/final_clean_split_data.csv", index=False)
    print(f"Clean dataset saved: {DATASET_FOLDER}/final_clean_split_data.csv")
    df = pd.read_csv(f"{DATASET_FOLDER}/final_clean_split_data.csv")
else:
    print("Dataset already split, skipping...")

Clean dataset saved: correlated_data_csv/final_clean_split_data.csv


# TRAIN MODELS PER CLUSTER

In [ ]:
results = []

for cluster_id in sorted(df["cluster"].unique()):
    print(f"\n - Training cluster {cluster_id}")

    # Select cluster and sort by time
    df_c = df[df["cluster"] == cluster_id].sort_values("timestamp")

    if "total_bikes_lag1" not in df_c.columns:
        df_c["total_bikes_lag1"] = df_c.groupby("station_code")["total_bikes_now"].shift(1)
        df_c["total_bikes_lag2"] = df_c.groupby("station_code")["total_bikes_now"].shift(2)
        df_c = df_c.dropna(subset=["total_bikes_lag1", "total_bikes_lag2"])

    FEATURES_LAG = FEATURES + ["total_bikes_lag1", "total_bikes_lag2"]

    # Select train/val/test using the split column
    train = df_c[df_c["split"] == "train"]
    val   = df_c[df_c["split"] == "val"]
    # test  = df_c[df_c["split"] == "test"]

    X_train, y_train = train[FEATURES_LAG], train[TARGET]
    X_val, y_val     = val[FEATURES_LAG], val[TARGET]
    # X_test, y_test   = test[FEATURES_LAG], test[TARGET]

    models = {
        "LinearRegression": LinearRegression(),
        "SGDRegressor": Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ),
    }

    # Train and evaluate on validation set
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        r2 = r2_score(y_val, preds)
        mae = mean_absolute_error(y_val, preds)

        results.append({
            "cluster": cluster_id,
            "model": name,
            "r2": r2,
            "mae": mae,
            "n_samples": len(df_c)
        })

        print(f"  {name:16s} → R^2={r2:.3f} | MAE={mae:.2f}")



 - Training cluster 0
  LinearRegression → R^2=0.938 | MAE=1.02
  SGDRegressor     → R^2=0.938 | MAE=1.02


KeyboardInterrupt: 

# COMPARE RESULTS

In [ ]:
results_df = pd.DataFrame(results)
results_df.sort_values(["cluster", "r2"], ascending=[True, False])

,cluster,model,r2,mae,n_samples
2,0,RandomForest,0.957164,0.840286,3457266
1,0,SGDRegressor,0.954185,0.909167,3457266
0,0,LinearRegression,0.954157,0.917324,3457266
5,1,RandomForest,0.976724,1.013360,3343550
3,1,LinearRegression,0.975355,1.094892,3343550
4,1,SGDRegressor,0.975321,1.102640,3343550


# GET BEST MODEL PER CLUSTER

In [ ]:
best_models = {}

for cluster_id in results_df["cluster"].unique():
    best = results_df[results_df["cluster"] == cluster_id].sort_values(
        "r2", ascending=False
    ).iloc[0]

    best_models[cluster_id] = best["model"]

print(best_models)

{np.int64(0): 'RandomForest', np.int64(1): 'RandomForest'}


# SAVE BEST MODEL PER CLUSTER

In [ ]:
for cluster_id in best_models:
    df_c = df[df["cluster"] == cluster_id]
    X = df_c[FEATURES]
    y = df_c[TARGET]

    model_name = best_models[cluster_id]

    if model_name == "LinearRegression":
        model = LinearRegression()
    elif model_name == "SGDRegressor":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ])
    else:
        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        )

    model.fit(X, y)
    joblib.dump(model, f"{MODEL_OUTPUT_FOLDER}/model_cluster_{cluster_id}.pkl")

print("Models saved per cluster")


Models saved per cluster


# QUICK TEST ON THE MODELS

In [ ]:
# LOAD THE MODELS
pred_models = {}
for i in range(2):
    pred_models[i] = joblib.load(f"{MODEL_OUTPUT_FOLDER}/model_cluster_{i}.pkl")

In [ ]:
test_clusters = sorted(df["cluster"].unique())

for cluster_id in test_clusters:
    df_c = df[df["cluster"] == cluster_id].sort_values("timestamp")
    
    X = df_c[FEATURES]
    y = df_c[TARGET]
    
    model_name = best_models[cluster_id]
    if model_name == "LinearRegression":
        model = LinearRegression()
    elif model_name == "SGDRegressor":
        model = Pipeline([("scaler", StandardScaler()),
                          ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))])
    else:
        model = RandomForestRegressor(n_estimators=200, max_depth=15,
                                      random_state=42, n_jobs=-1)
    
    model = pred_models[cluster_id]
    preds = model.predict(X)
    
    plt.figure(figsize=(12,4))
    plt.plot(df_c["timestamp"], y, label="Actual", alpha=0.7)
    plt.plot(df_c["timestamp"], preds, label="Predicted", alpha=0.7)
    plt.title(f"Cluster {cluster_id} - {model_name}")
    plt.xlabel("Timestamp")
    plt.ylabel("Total Bikes Now")
    plt.legend()
    plt.savefig(PLOT_FOLDER / f"cluster_{cluster_id}_{model_name}.png")
    plt.close()

C:\Users\emirh\AppData\Local\Temp\ipykernel_43420\1190566852.py:33: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(PLOT_FOLDER / f"cluster_{cluster_id}_{model_name}.png")
C:\Users\emirh\AppData\Local\Temp\ipykernel_43420\1190566852.py:33: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(PLOT_FOLDER / f"cluster_{cluster_id}_{model_name}.png")


In [ ]:
for cluster_id in sorted(df["cluster"].unique()):
    stations_in_cluster = df.loc[
        df["cluster"] == cluster_id, "station_code"
    ].unique()

    for station_code in stations_in_cluster:
        df_station = (
            df[df["station_code"] == station_code]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )
        print(f"Plotting cluster {cluster_id}")

        stations_in_cluster = df[
            df["cluster"] == cluster_id
        ]["station_code"].unique()

        station_code = np.random.choice(stations_in_cluster)
        print(f"Station: {station_code}")

        df_station = (
            df[df["station_code"] == station_code]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        X = df_station[FEATURES]
        y = df_station[TARGET]

        model = models[cluster_id]

        preds = model.predict(X)

        plt.figure(figsize=(14, 5))
        plt.plot(df_station["timestamp"], y, label="Actual", alpha=0.8)
        plt.plot(df_station["timestamp"], preds, label="Predicted", alpha=0.8)

        plt.title(
            f"Cluster {cluster_id} | Station {station_code}\n"
            f"Model: {type(model).__name__}"
        )
        plt.xlabel("Timestamp")
        plt.ylabel("Total Bikes Now")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{PLOT_FOLDER}/cluster_{cluster_id}_station_{station_code}.png")
        plt.close()

Plotting cluster 0


KeyboardInterrupt: 

Error in callback <function _draw_all_if_interactive at 0x000002DEE5BCB880> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 